In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!pip install nltk --quiet

import re
import pandas as pd
import numpy as np
import nltk

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

print('All libraries imported!')

All libraries imported!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [5]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This...
  """
  # Convert the input to small/lower order.
  data = dataset.lower()
  # Remove URLs
  data =re.sub(r'http\S+|www\S+|https\S+', '', data)
  # Remove emojis
  data = data.encode('ascii','ignore').decode('ascii')
  # Remove all other unwanted characters.
  data =re.sub(r'[^a-z\s]', '', data)
  # Create tokens.
  tokens = data.split()
  # Remove stopwords:
  stop_words = set(stopwords.words('english'))
  tokens =[word for word in tokens if word not in stop_words]
  if rule == "lemmatize":
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Pick between lemmatize or stem")


  return " ".join(tokens)


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [12]:
df = pd.read_csv('/content/drive/MyDrive/aayushma/trum_tweet_sentiment_analysis.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

Shape: (1850123, 2)
Columns: ['text', 'Sentiment']


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [15]:
print('Label column sample values:')
print(df['Sentiment'].value_counts().head(10))
print()
print('Null values:')
print(df[['text', 'Sentiment']].isnull().sum())

Label column sample values:
Sentiment
0    1244211
1     605912
Name: count, dtype: int64

Null values:
text         0
Sentiment    0
dtype: int64


In [17]:
df = df[['text', 'Sentiment']].dropna().sample(10000, random_state=42).reset_index(drop=True)
df['cleaned_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x))
print('BEFORE', df['text'][0])
print('AFTER', df['cleaned_text'][0])

BEFORE RT @EthandeMarsi: I don't understand why adults would be mean to a kid! We're the same age! @realDonaldTrump , I'll be friends w/Barr https://t.co/1zFUGhrU0s
AFTER rt ethandemarsi dont understand adult would mean kid age realdonaldtrump ill friend wbarr


In [18]:
X=df['cleaned_text']
Y=df['Sentiment']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f'Training samples:{len(X_train)}')
print(f'Testing samples:{len(X_test)}')

Training samples:8000
Testing samples:2000


In [19]:
tfidf=TfidfVectorizer(max_features=5000)
X_train_tfidf=tfidf.fit_transform(X_train)
X_test_tfidf=tfidf.transform(X_test)

print('Training matrix shape:', X_train_tfidf.shape)
print('Testing matrix shape:', X_test_tfidf.shape)
print(f'\nEach tweet is now a row of {X_train_tfidf.shape[1]}numbers.')

Training matrix shape: (8000, 5000)
Testing matrix shape: (2000, 5000)

Each tweet is now a row of 5000numbers.


In [21]:
model=LogisticRegression(max_iter=1000,random_state=42)
model.fit(X_train_tfidf,Y_train)

print("Model trained successfully.")

Model trained successfully.


In [26]:
Y_pred=model.predict(X_test_tfidf)

print('CLASSIFICATION REPORT')
print('='*55)
print(classification_report(Y_test,Y_pred))

accuracy=(Y_pred == Y_test).mean()
print('Accuracy:{accuracy:.2%}')

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.80      0.96      0.87      1341
           1       0.86      0.50      0.63       659

    accuracy                           0.81      2000
   macro avg       0.83      0.73      0.75      2000
weighted avg       0.82      0.81      0.79      2000

Accuracy:{accuracy:.2%}


In [28]:
def predict_sentiment(text):
    cleaned = text_cleaning_pipeline(text)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    print(f'Input: {text}')
    print(f'Cleaned: {cleaned}')
    print(f'Prediction: {prediction}')
    print()

predict_sentiment('Trump is doing a great job for America!')
predict_sentiment('This policy is a complete disaster and very unfair.')
predict_sentiment('The president signed a new trade deal today.')

Input: Trump is doing a great job for America!
Cleaned: trump great job america
Prediction: 1

Input: This policy is a complete disaster and very unfair.
Cleaned: policy complete disaster unfair
Prediction: 0

Input: The president signed a new trade deal today.
Cleaned: president signed new trade deal today
Prediction: 1

